# GeoLift market selection — Python walkthrough (`geolift_fast`)

This notebook runs **the same example as Meta GeoLift's R market-selection walkthrough** —
on the canonical `GeoLift_Test` dataset (40 real US cities x 105 days) — and shows that the
Python port [`geolift_fast`](README.md) produces the **same results as R**, while running
~10-14x faster (see [`../REPORT.md`](../REPORT.md) §7).

### R -> Python, step for step

| GeoLift (R) | `geolift_fast` (Python) |
|---|---|
| `GeoDataRead(data, date_id, location_id, Y_id, …)` | `Panel.from_long_csv(path)` / `Panel.from_long_df(df)` |
| `GeoLiftMarketSelection(…)$PowerCurves` | `power_curves(panel, combos, …)` |
| `GeoLiftMarketSelection(…)$BestMarkets` | `best_markets(power_curves_df)` |
| `N = c(2)` (treatment size = 2) | `all_pairs(panel, size=2)` |

The R walkthrough this mirrors (reproduced by [`../exploration/scripts/build_city_example.R`](../exploration/scripts/build_city_example.R)):

```r
library(GeoLift)
data(GeoLift_Test)                                   # 40 US cities x 105 days
GeoTestData <- GeoDataRead(data = GeoLift_Test, date_id = "date",
                          location_id = "location", Y_id = "Y",
                          X = c(), format = "yyyy-mm-dd", summary = FALSE)
MarketSelection <- GeoLiftMarketSelection(data = GeoTestData,
                          treatment_periods = c(14), N = c(2),
                          Y_id = "Y", location_id = "location", time_id = "time",
                          effect_size = seq(-0.10, 0.10, 0.05), ns = 1000)
MarketSelection$BestMarkets
```

## 0. Install

From GitHub (works on a GCP VM), or `pip install -e .` from the repo root for local dev:

```bash
pip install "git+https://github.com/moraisnara/geolift-port.git@v0.1.0"
```

Dependencies (`numpy`, `pandas`, `scipy`, `osqp`) resolve automatically. This notebook lives
inside the repo, so it imports the package directly and reads the example data from
`../exploration/`.

In [1]:
import pandas as pd
from geolift_fast import Panel, power_curves, best_markets, all_pairs

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

## 1. Read the panel  —  `GeoDataRead` -> `Panel.from_long_csv`

GeoLift's `GeoDataRead` ingests a long/tidy panel (one row per `location` x `date`) and assigns
an integer `time = 1...T` in date order. `Panel.from_long_csv` does the same — lowercasing
locations and ordering time by sorted unique date — so the wide outcome matrix matches R
cell-for-cell. We use the raw `GeoLift_Test` panel exported to CSV.

In [2]:
panel = Panel.from_long_csv("../exploration/data/geolift_test_panel.csv")
print(f"{len(panel.locations)} cities x {panel.T} periods")
panel.locations[:8]

40 cities x 105 periods


['atlanta',
 'austin',
 'baltimore',
 'baton rouge',
 'boston',
 'chicago',
 'cincinnati',
 'cleveland']

## 2. Candidate test markets  —  `N = c(2)`

With `N = c(2)`, R's `GeoLiftMarketSelection` does **not** brute-force all C(40,2)=780 pairs — it
uses a correlation-based selector (`stochastic_market_selector`) to shortlist a manageable set of
size-2 candidates. To reproduce R's example exactly, we evaluate the **same combos R chose** (read
from R's output). For an open-ended search you'd instead pass `all_pairs(panel, size=2)` or your
own list of lists.

In [3]:
r_pc = pd.read_csv("../exploration/results/citylift_R_powercurves.csv")
r_pc["location"] = r_pc["location"].str.lower()
combos = [s.split(", ") for s in pd.unique(r_pc.location)]   # the size-2 candidates R selected
print(f"{len(combos)} candidate city pairs")
combos[:5]

31 candidate city pairs


[['atlanta', 'chicago'],
 ['atlanta', 'nashville'],
 ['austin', 'oakland'],
 ['baltimore', 'philadelphia'],
 ['baton rouge', 'cincinnati']]

## 3. Power simulation  —  `GeoLiftMarketSelection(…)$PowerCurves` -> `power_curves(…)`

`power_curves` runs GeoLift's power simulation for every (combo x test-window x effect size) cell:
it fits the fixed-effects synthetic control on the pre-period, computes the ATT, detected lift and
scaled-L2 imbalance, and gets the conformal "iid" permutation p-value. It returns the same tidy
table as R's `$PowerCurves`. Same knobs as the R call: `treatment_periods=14`,
`effect_sizes=seq(-0.10, 0.10, 0.05)`, `ns=1000`, `seed=42`.

In [4]:
pc = power_curves(
    panel, combos,
    treatment_periods=14,
    effect_sizes=[-0.10, -0.05, 0.0, 0.05, 0.10],
    ns=1000,                # conformal permutations (Meta default)
    alpha=0.10,
    seed=42,                # statistics deterministic given the seed
)
# columns: location, duration, EffectSize, pvalue, power, AvgATT, AvgDetectedLift, AvgScaledL2Imbalance
pc.head(10)

,location,duration,EffectSize,pvalue,power,AvgATT,AvgDetectedLift,AvgScaledL2Imbalance
0,"atlanta, chicago",14,-0.10,0.013,1,-343.501587,-0.099594,0.432280
1,"atlanta, chicago",14,-0.05,0.170,0,-170.973015,-0.049572,0.432280
2,"atlanta, chicago",14,0.00,0.438,0,1.555556,0.000451,0.432280
3,"atlanta, chicago",14,0.05,0.118,0,174.084128,0.050474,0.432280
4,"atlanta, chicago",14,0.10,0.014,1,346.612699,0.100496,0.432280
5,"atlanta, nashville",14,-0.10,0.032,1,-37.593797,-0.011182,0.493571
6,"atlanta, nashville",14,-0.05,0.040,1,147.102631,0.043753,0.493571
7,"atlanta, nashville",14,0.00,0.010,1,331.799060,0.098687,0.493571
8,"atlanta, nashville",14,0.05,0.001,1,516.495489,0.153622,0.493571
9,"atlanta, nashville",14,0.10,0.001,1,701.191917,0.208556,0.493571


## 4. Rank the markets  —  `GeoLiftMarketSelection(…)$BestMarkets` -> `best_markets(…)`

`best_markets` reproduces GeoLift's `$BestMarkets`: per city pair it picks the **minimum
detectable effect (MDE)** — the significant effect size closest to zero (considering both
directions, GeoLift's rule) — then orders pairs by R's composite of three sub-ranks (`|MDE|`,
`power`, `abs_lift_in_zero`). The cities at the top are the most sensitive test designs.

In [5]:
ranking = best_markets(pc)   # columns: rank, location, duration, MDE, AvgDetectedLift, abs_lift_in_zero
ranking.head(10)

,rank,location,duration,MDE,AvgDetectedLift,abs_lift_in_zero
0,1,"atlanta, chicago",14,0.10,0.100496,0.000
1,1,"jacksonville, milwaukee",14,-0.05,-0.046519,0.003
2,3,"columbus, new york",14,0.10,0.096965,0.003
3,4,"dallas, denver",14,-0.05,-0.057211,0.007
4,4,"dallas, memphis",14,0.10,0.105769,0.006
5,6,"jacksonville, minneapolis",14,-0.10,-0.108554,0.009
6,7,"austin, oakland",14,0.05,0.027275,0.023
7,7,"cleveland, denver",14,-0.05,-0.072763,0.023
8,7,"dallas, washington",14,-0.10,-0.079693,0.020
9,10,"kansas city, milwaukee",14,-0.10,-0.076651,0.023


## 5. Does it match R?  —  same example, same results

Now the point of the exercise: line up Python's per-cell output against R's `$PowerCurves` on the
**exact same combos**. The deterministic estimators (ATT, scaled-L2 imbalance, detected lift) match
R to **solver tolerance**; the significance flag agrees on **~99%** of cells. (GeoLift's p-value is
a *permutation* p-value drawn from RNG; R and Python use independent generators, so the handful of
flips are alpha-boundary cells where the Monte-Carlo p-value straddles 0.10 — never a structural
disagreement.)

In [6]:
j = r_pc.merge(pc, on=["location", "duration", "EffectSize"], suffixes=("_R", "_py"))
print(f"cells compared            : {len(j)}")
print(f"significance agreement    : {100*(j.power_R == j.power_py).mean():.1f}%")
print(f"ATT        max |R - py|   : {(j.AvgATT_R - j.AvgATT_py).abs().max():.2e}")
print(f"scaled-L2  max |R - py|   : {(j.AvgScaledL2Imbalance_R - j.AvgScaledL2Imbalance_py).abs().max():.2e}")
print(f"det. lift  max |R - py|   : {(j.AvgDetectedLift_R - j.AvgDetectedLift_py).abs().max():.2e}")

cells compared            : 155
significance agreement    : 98.7%
ATT        max |R - py|   : 4.28e-04
scaled-L2  max |R - py|   : 2.10e-08
det. lift  max |R - py|   : 1.86e-07


And the selected markets: Python and R pick the **same set** of best markets with the **same MDE
magnitude** for every pair, and identical top-5 / top-10 sets. (A few pairs are significant at both
+0.05 and -0.05; which sign is reported as "the" MDE, and the exact integer rank within a tie group,
depend on a third-decimal tiebreak — neither changes which markets are chosen.) Here are R's and
Python's top-10 side by side:

In [7]:
r_bm = pd.read_csv("../exploration/results/citylift_R_bestmarkets.csv")
r_bm["location"] = r_bm["location"].str.lower()

py_set, r_set = set(ranking.location), set(r_bm.location)
print(f"best-market set identical : {py_set == r_set}  ({len(py_set)} pairs each)")
py_mde = dict(zip(ranking.location, ranking.MDE))
r_mde = dict(zip(r_bm.location, r_bm.EffectSize))
common = py_set & r_set
print(f"MDE magnitude agreement   : {100*sum(abs(py_mde[l])==abs(r_mde[l]) for l in common)/len(common):.0f}%")

side = pd.concat([
    r_bm.sort_values("rank")["location"].head(10).reset_index(drop=True).rename("R_best"),
    ranking["location"].head(10).reset_index(drop=True).rename("Python_best"),
], axis=1)
side.index = side.index + 1
side

best-market set identical : True  (30 pairs each)
MDE magnitude agreement   : 100%


,R_best,Python_best
1,"jacksonville, milwaukee","atlanta, chicago"
2,"atlanta, chicago","jacksonville, milwaukee"
3,"columbus, new york","columbus, new york"
4,"dallas, denver","dallas, denver"
5,"dallas, memphis","dallas, memphis"
6,"austin, oakland","jacksonville, minneapolis"
7,"jacksonville, minneapolis","austin, oakland"
8,"cleveland, denver","cleveland, denver"
9,"dallas, washington","dallas, washington"
10,"kansas city, milwaukee","kansas city, milwaukee"


## 6. Inspect a chosen market

Slice the power curve for the top-ranked pair to see how power rises with effect size — the same
diagnostic you'd read off `MarketSelection$PowerCurves` in R.

In [8]:
top = ranking.loc[0, "location"]
pc[pc.location == top][["EffectSize", "pvalue", "power", "AvgATT", "AvgDetectedLift", "AvgScaledL2Imbalance"]]

,EffectSize,pvalue,power,AvgATT,AvgDetectedLift,AvgScaledL2Imbalance
0,-0.10,0.013,1,-343.501587,-0.099594,0.43228
1,-0.05,0.170,0,-170.973015,-0.049572,0.43228
2,0.00,0.438,0,1.555556,0.000451,0.43228
3,0.05,0.118,0,174.084128,0.050474,0.43228
4,0.10,0.014,1,346.612699,0.100496,0.43228


## Lower-level building blocks (optional)

For custom pipelines the per-combo engine is exposed directly. `simulate_combo` fits one combo and
returns its cells across all effect sizes (reusing the effect-size-invariant pre-period fit —
GeoLift's "lever 2"). `ComboFit`, `scm_weights`, `conformal_resids`, and `conformal_pval` give you
the SCM weights, residuals, and permutation p-value individually.

```python
import numpy as np
from geolift_fast import simulate_combo

rng = np.random.default_rng(42)
cells = simulate_combo(panel, ["chicago", "portland"], tp=14,
                       effect_sizes=[0.0, 0.05, 0.10], ns=1000, rng=rng, alpha=0.10)
```

The full R-vs-Python agreement on this example is computed by
[`compare_city_example.py`](compare_city_example.py) -> `exploration/results/citylift_python_compare.json`,
and the scaling/speedup story is in [`../REPORT.md`](../REPORT.md) §7.